# Ablations and patch length (Reviewer 2, comments 1 and 2)

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

At 1000 evaluations, on all four systems, five folds and three seeds:

- `ABL-UMDA-imp`: UMDA4GI learning only from improving variants instead of all valid ones;
- `ABL-UMDA-k1`, `ABL-UMDA-k3`: UMDA4GI with $k_{max}$ = 1 and 3 (the default is 2);
- `ABL-FI-2`, `ABL-GP-2`: First-Improvement and GP limited to at most two edits per patch.

Wilcoxon signed-rank tests paired by fold and seed; Holm's correction over the 60 tests of the ablations (improvement, gap and patch length).

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

In [2]:
abl = pd.read_csv(OUT / 'ablations.csv')
display(abl)
print('significant after Holm:')
display(abl[abl.p_holm < 0.05])

,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
0,minisat_hack,1000,ABL-UMDA-imp,UMDA,improvement,15,-3.51e-01,-0.39,0.52,8.90e-01,1.00e+00
1,minisat_hack,1000,ABL-UMDA-imp,UMDA,gap,15,1.73e+01,21.23,0.49,5.24e-01,1.00e+00
2,minisat_hack,1000,ABL-UMDA-imp,UMDA,patch_length,15,2.00e+00,2.00,0.43,4.21e-01,1.00e+00
3,pngOptim,1000,ABL-UMDA-imp,UMDA,improvement,15,5.36e-02,-0.12,0.26,3.53e-02,1.00e+00
4,pngOptim,1000,ABL-UMDA-imp,UMDA,gap,15,9.97e-02,-0.03,0.19,5.37e-03,2.52e-01
5,pngOptim,1000,ABL-UMDA-imp,UMDA,patch_length,15,1.00e+00,2.00,0.58,5.35e-01,1.00e+00
6,sat4j,1000,ABL-UMDA-imp,UMDA,improvement,15,-8.72e-01,0.04,0.53,5.24e-01,1.00e+00
7,sat4j,1000,ABL-UMDA-imp,UMDA,gap,15,4.87e+00,9.91,0.60,3.89e-01,1.00e+00
8,sat4j,1000,ABL-UMDA-imp,UMDA,patch_length,15,2.00e+00,2.00,0.47,7.26e-01,1.00e+00
9,weka,1000,ABL-UMDA-imp,UMDA,improvement,15,-2.82e-01,-0.66,0.37,1.88e-01,1.00e+00


significant after Holm:


,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
14,minisat_hack,1000,ABL-UMDA-k1,UMDA,patch_length,15,1.00,2.00,0.90,5.71e-04,3.31e-02
27,pngOptim,1000,ABL-UMDA-k3,UMDA,improvement,15,0.17,-0.12,0.14,6.10e-05,3.66e-03
28,pngOptim,1000,ABL-UMDA-k3,UMDA,gap,15,0.26,-0.03,0.13,1.83e-04,1.08e-02
38,minisat_hack,1000,ABL-FI-2,FI,patch_length,15,2.00,11.00,1.00,6.42e-04,3.59e-02
44,sat4j,1000,ABL-FI-2,FI,patch_length,15,2.00,7.00,1.00,6.24e-04,3.56e-02
47,weka,1000,ABL-FI-2,FI,patch_length,15,2.00,5.00,0.95,7.72e-04,4.17e-02
50,minisat_hack,1000,ABL-GP-2,GP,patch_length,15,2.00,6.00,0.97,7.09e-04,3.90e-02
53,pngOptim,1000,ABL-GP-2,GP,patch_length,15,2.00,3.00,0.89,7.77e-04,4.17e-02


## UMDA4GI learning only from improving variants: generations with nothing to learn from

Runs of `ABL-UMDA-imp` in which at least one generation had no improving variant (the previous distribution is then kept).

In [3]:
imp = df[(df.group == 'ABL-UMDA-imp') & (df.checkpoint == 1000)]
display(imp.groupby('software').gens_without_valid.agg(runs='size', runs_with_such_generations=lambda s: (s > 0).sum()))

,runs,runs_with_such_generations
software,,
minisat_hack,15,0
pngOptim,15,2
sat4j,15,0
weka,15,2


## Patch length of the two-edit baselines

Median length of the returned patch at 1000 evaluations.

In [4]:
last = df[df.checkpoint == 1000]
display(last[last.group.isin(['FI', 'ABL-FI-2', 'GP', 'ABL-GP-2', 'UMDA'])]
        .pivot_table(index='software', columns='group', values='patch_length', aggfunc='median'))

group,ABL-FI-2,ABL-GP-2,FI,GP,UMDA
software,,,,,
minisat_hack,2.0,2.0,11.0,6.0,2.0
pngOptim,2.0,2.0,5.0,3.0,2.0
sat4j,2.0,2.0,7.0,4.0,2.0
weka,2.0,2.0,5.0,2.0,2.0


## Limiting FI and GP to two edits: change in the median gap

In [5]:
c = cells[cells.budget == 1000].set_index(['software', 'group'])['gap_median']
rows = [{'software': sw, 'algorithm': g, 'gap': c[(sw, g)], 'gap with two edits': c[(sw, f'ABL-{g}-2')]}
        for sw in ['minisat_hack', 'sat4j', 'weka', 'pngOptim'] for g in ['FI', 'GP']]
t = pd.DataFrame(rows)
t['lower'] = t['gap with two edits'] < t['gap']
display(t)

,software,algorithm,gap,gap with two edits,lower
0,minisat_hack,FI,27.30,2.37e+01,True
1,minisat_hack,GP,28.99,8.23e+00,True
2,sat4j,FI,16.17,8.87e+00,True
3,sat4j,GP,7.37,2.21e+00,True
4,weka,FI,2.16,-9.60e-03,True
5,weka,GP,2.47,2.96e+00,False
6,pngOptim,FI,0.02,1.31e-01,False
7,pngOptim,GP,0.13,2.15e-01,False


## UMDA4GI against the length-controlled baselines

Holm's correction over these 16 tests.

In [6]:
display(pd.read_csv(OUT / 'length_controlled.csv'))

,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
0,minisat_hack,1000,UMDA,ABL-FI-2,improvement,15,-0.39,-2.30e+01,0.34,9.46e-02,1.00
1,minisat_hack,1000,UMDA,ABL-FI-2,gap,15,21.23,2.37e+01,0.57,3.59e-01,1.00
2,pngOptim,1000,UMDA,ABL-FI-2,improvement,15,-0.12,1.34e-02,0.59,7.62e-01,1.00
3,pngOptim,1000,UMDA,ABL-FI-2,gap,15,-0.03,1.31e-01,0.68,2.08e-01,1.00
4,sat4j,1000,UMDA,ABL-FI-2,improvement,15,0.04,-2.20e+00,0.40,5.24e-01,1.00
5,sat4j,1000,UMDA,ABL-FI-2,gap,15,9.91,8.87e+00,0.48,8.47e-01,1.00
6,weka,1000,UMDA,ABL-FI-2,improvement,15,-0.66,-1.70e+00,0.47,9.34e-01,1.00
7,weka,1000,UMDA,ABL-FI-2,gap,15,0.44,-9.60e-03,0.43,7.62e-01,1.00
8,minisat_hack,1000,UMDA,ABL-GP-2,improvement,15,-0.39,-1.82e+01,0.36,8.33e-02,1.00
9,minisat_hack,1000,UMDA,ABL-GP-2,gap,15,21.23,8.23e+00,0.48,8.90e-01,1.00
